# Layer 1 visualizations

In [ ]:
%pip install matplotlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected configs/config.yaml in project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f) or {}

OUTPUT_DIR = PROJECT_ROOT / CONFIG.get("paths", {}).get("output_dir", "output")
FIG_DIR = OUTPUT_DIR / CONFIG.get("visualization", {}).get("figures_subdir", "08_layer1_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIG_DIR:", FIG_DIR)


In [ ]:
frame_index = pd.read_csv(OUTPUT_DIR / "01_ssl_frames" / "frame_index.csv")
segment_summary = pd.read_csv(OUTPUT_DIR / CONFIG.get("segmentation", {}).get("output_subdir", "02_segments") / "segment_summary.csv")
segments = pd.read_csv(OUTPUT_DIR / CONFIG.get("segmentation", {}).get("output_subdir", "02_segments") / "segments.csv")
kstudy = pd.read_csv(OUTPUT_DIR / CONFIG.get("clustering", {}).get("kstudy_subdir", "04_kstudy") / "kstudy_results.csv")
global_tokens = pd.read_csv(OUTPUT_DIR / CONFIG.get("tokens", {}).get("output_subdir", "05_tokens") / "global_token_frequencies.csv")
unit_inventory = pd.read_csv(OUTPUT_DIR / CONFIG.get("dictionary", {}).get("output_subdir", "06_dictionary") / "unit_inventory.csv")
unit_prototypes = pd.read_csv(OUTPUT_DIR / CONFIG.get("dictionary", {}).get("output_subdir", "06_dictionary") / "unit_prototypes.csv")
audit_dir = OUTPUT_DIR / CONFIG.get("audits", {}).get("output_subdir", "07_audits")
unit_farm_bias = pd.read_csv(audit_dir / "unit_farm_bias.csv")
unit_zone_bias = pd.read_csv(audit_dir / "unit_zone_bias.csv")
unit_microphone_bias = pd.read_csv(audit_dir / "unit_microphone_bias.csv")
unit_placement_bias = pd.read_csv(audit_dir / "unit_placement_bias.csv")

print("Loaded all Layer 1 outputs.")


In [ ]:
summary_metrics = {
    "n_clips_ssl": int(len(frame_index)),
    "n_clips_segmented": int(len(segment_summary)),
    "n_segments_total": int(len(segments)),
    "mean_clip_duration_sec": float(frame_index["duration_sec"].mean()),
    "median_clip_duration_sec": float(frame_index["duration_sec"].median()),
    "mean_segments_per_clip": float(segment_summary["final_segment_count"].mean()),
    "median_segments_per_clip": float(segment_summary["final_segment_count"].median()),
    "single_segment_clips": int((segment_summary["final_segment_count"] == 1).sum()),
    "single_segment_clip_pct": float((segment_summary["final_segment_count"] == 1).mean() * 100.0),
    "max_segments_single_clip": int(segment_summary["final_segment_count"].max()),
    "n_units": int(unit_inventory["unit_id"].nunique()),
    "prototypes_total": int(len(unit_prototypes)),
    "prototypes_per_unit": float(len(unit_prototypes) / max(1, unit_inventory["unit_id"].nunique())),
}
summary_df = pd.DataFrame([summary_metrics]).T.reset_index()
summary_df.columns = ["metric", "value"]
summary_df.to_csv(FIG_DIR / "layer1_summary_metrics.csv", index=False)
summary_df


In [ ]:
def save_current_figure(filename: str, dpi: int = 220):
    out_path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return out_path


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(frame_index["duration_sec"], bins=40)
plt.xlabel("Clip duration (sec)")
plt.ylabel("Number of clips")
plt.title("Layer 1 input clip duration distribution")
save_current_figure("fig01_clip_duration_distribution.png")
plt.show()


In [ ]:
segment_counts = segment_summary["final_segment_count"]
bins = np.arange(segment_counts.min(), segment_counts.max() + 2) - 0.5
plt.figure(figsize=(9, 5))
plt.hist(segment_counts, bins=bins)
plt.xlabel("Segments per clip")
plt.ylabel("Number of clips")
plt.title("Unsupervised segmentation output: segments per clip")
save_current_figure("fig02_segments_per_clip.png")
plt.show()
segment_counts.describe()


In [ ]:
kstudy_sorted = kstudy.sort_values("k")
plt.figure(figsize=(8, 5))
plt.plot(kstudy_sorted["k"], kstudy_sorted["mean_pairwise_ari"], marker="o")
plt.xlabel("K")
plt.ylabel("Mean pairwise ARI across seeds")
plt.title("K-study stability across tested K values")
for _, row in kstudy_sorted.iterrows():
    plt.annotate(f"{row['mean_pairwise_ari']:.3f}", (row["k"], row["mean_pairwise_ari"]), textcoords="offset points", xytext=(0, 7), ha="center")
save_current_figure("fig03_kstudy_stability_ari.png")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(kstudy_sorted["k"], kstudy_sorted["mean_singleton_clusters"], marker="o", label="Mean singleton clusters")
plt.plot(kstudy_sorted["k"], kstudy_sorted["mean_min_cluster_size"], marker="o", label="Mean min cluster size")
plt.xlabel("K")
plt.ylabel("Cluster fragmentation summary")
plt.title("K-study fragmentation trade-off")
plt.legend()
save_current_figure("fig04_kstudy_fragmentation.png")
plt.show()


In [ ]:
top_units = global_tokens.sort_values("segment_count", ascending=False).head(15).copy()
plt.figure(figsize=(10, 5))
plt.bar(top_units["unit_id"], top_units["segment_count"])
plt.xlabel("Unit ID")
plt.ylabel("Segment count")
plt.title("Top 15 most frequent discovered units")
save_current_figure("fig05_top15_unit_prevalence.png")
plt.show()
top_units[["unit_id", "segment_count", "clip_count", "segment_proportion", "clip_proportion"]]


In [ ]:
u = unit_inventory.copy()
plt.figure(figsize=(8, 6))
plt.scatter(u["median_duration_sec"], u["segment_count"])
plt.xlabel("Median segment duration (sec)")
plt.ylabel("Segment count")
plt.title("Discovered units: duration vs prevalence")
to_annotate = u.sort_values("segment_count", ascending=False).head(8)
for _, row in to_annotate.iterrows():
    plt.annotate(row["unit_id"], (row["median_duration_sec"], row["segment_count"]), textcoords="offset points", xytext=(4, 4))
save_current_figure("fig06_unit_duration_vs_prevalence.png")
plt.show()


In [ ]:
bias_tables = {
    "farm": unit_farm_bias,
    "barn_zone": unit_zone_bias,
    "microphone": unit_microphone_bias,
    "mic_placement_context": unit_placement_bias,
}
bias_summary_rows = []
for field_name, df in bias_tables.items():
    bias_summary_rows.append({
        "field_name": field_name,
        "flagged_units": int(df["flag_context_locked"].fillna(False).astype(bool).sum()),
        "high_bias_units": int((df["bias_level"].astype(str).str.lower() == "high").sum()),
        "median_dominant_context_proportion": float(df["dominant_context_proportion"].median()),
        "max_dominant_context_proportion": float(df["dominant_context_proportion"].max()),
    })
bias_summary = pd.DataFrame(bias_summary_rows)
bias_summary.to_csv(FIG_DIR / "bias_summary_by_context.csv", index=False)

x = np.arange(len(bias_summary))
width = 0.35
plt.figure(figsize=(9, 5))
plt.bar(x - width/2, bias_summary["flagged_units"], width=width, label="Flagged units")
plt.bar(x + width/2, bias_summary["high_bias_units"], width=width, label="High-bias units")
plt.xticks(x, bias_summary["field_name"], rotation=15)
plt.ylabel("Number of units")
plt.title("Bias-audit summary across context types")
plt.legend()
save_current_figure("fig07_bias_summary_by_context.png")
plt.show()
bias_summary


In [ ]:
combined_bias = pd.concat(
    [
        unit_farm_bias.assign(field_name="farm"),
        unit_zone_bias.assign(field_name="barn_zone"),
        unit_microphone_bias.assign(field_name="microphone"),
        unit_placement_bias.assign(field_name="mic_placement_context"),
    ],
    ignore_index=True
)
combined_bias = combined_bias[combined_bias["segment_count"] >= 8].copy()
combined_bias["unit_field"] = combined_bias["unit_id"] + " | " + combined_bias["field_name"]
top_locked = (
    combined_bias.sort_values(["dominant_context_proportion", "segment_count"], ascending=[False, False])
    .drop_duplicates(["unit_id", "field_name"])
    .head(12)
    .copy()
)
plt.figure(figsize=(11, 6))
plt.barh(top_locked["unit_field"], top_locked["dominant_context_proportion"])
plt.xlabel("Dominant context proportion")
plt.ylabel("Unit | audited field")
plt.title("Top context-locked discovered units")
plt.gca().invert_yaxis()
save_current_figure("fig08_top_context_locked_units.png")
plt.show()
top_locked[[
    "unit_id",
    "field_name",
    "segment_count",
    "clip_count",
    "dominant_context_value",
    "dominant_context_proportion",
    "dominant_sub_category_final",
]]


In [ ]:
caption_lines = [
    "# Layer 1 figure notes",
    "",
    "## fig01_clip_duration_distribution.png",
    "- Input clip duration distribution before segmentation.",
    "- Supports frame-level SSL extraction rather than direct clip pooling.",
    "",
    "## fig02_segments_per_clip.png",
    f"- Segmentation output across {len(segment_summary)} clips.",
    f"- {(segment_summary['final_segment_count'] == 1).sum()} clips ({(segment_summary['final_segment_count'] == 1).mean() * 100:.1f}%) remained single-segment, so this first segmentation pass is conservative.",
    "",
    "## fig03_kstudy_stability_ari.png",
    "- Mean pairwise ARI across seeds for each tested K.",
    "- Main figure supporting stability-based provisional K selection.",
    "",
    "## fig04_kstudy_fragmentation.png",
    "- Larger K values create more singleton clusters and smaller minimum cluster sizes.",
    "",
    "## fig05_top15_unit_prevalence.png",
    "- Frequency distribution of the most common discovered units.",
    "",
    "## fig06_unit_duration_vs_prevalence.png",
    "- Relates temporal footprint to prevalence of discovered units.",
    "",
    "## fig07_bias_summary_by_context.png",
    "- Number of flagged and high-bias units for each audited context field.",
    "",
    "## fig08_top_context_locked_units.png",
    "- Strongest context-locked units. These are audit flags, not biological conclusions.",
]
caption_path = FIG_DIR / "layer1_figure_notes.md"
caption_path.write_text("\n".join(caption_lines), encoding="utf-8")
print(f"Saved: {caption_path}")
